# EDA Kesehatan

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "hasil_cleaning_final" / "03_kesehatan_clean.csv"
if not DATA.is_file():
    raise FileNotFoundError("Jalankan cleaning.ipynb terlebih dahulu.")
OUTPUT = ROOT / "hasil_analisis_kesehatan"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Sumber:", DATA)


In [ ]:
df = pd.read_csv(DATA)
print("Ukuran:", df.shape)
print("Kolom:", df.columns.tolist())
display(df.head())


In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
indikator = ["resp_admissions_under5", "resp_admissions_over60", "asthma_emergency_cases", "copd_admissions"]
for kolom in indikator:
    df[kolom] = pd.to_numeric(df[kolom], errors="coerce")
print("Tanggal tidak valid:", df["date"].isna().sum())
print("Baris sebelum 2024:", (df["date"] < "2024-01-01").sum())
data = df[df["date"] >= "2024-01-01"].copy()
print("Baris untuk analisis:", len(data))


## Cek missing dan duplikat

In [ ]:
audit = pd.DataFrame({"missing": data[indikator].isna().sum(), "persen_missing": (data[indikator].isna().mean()*100).round(2)})
display(audit)
print("Duplikat rumah sakit-tanggal:", data.duplicated(["hospital_id", "date"]).sum())
print("Rumah sakit unik:", data["hospital_id"].nunique())
print("Rentang:", data["date"].min(), "sampai", data["date"].max())


## Ringkasan kasus

In [ ]:
kpi = pd.DataFrame({"indikator": indikator, "total_kasus": [data[k].sum(min_count=1) for k in indikator], "rata_rata_per_catatan": [data[k].mean() for k in indikator], "catatan_terisi": [data[k].count() for k in indikator]})
display(kpi)
kpi.to_csv(OUTPUT / "01_kpi_kesehatan.csv", index=False)


## Tren harian

In [ ]:
harian = data.groupby("date", as_index=False)[indikator].sum(min_count=1)
display(harian.head())
harian.to_csv(OUTPUT / "02_tren_harian.csv", index=False)
plt.figure(figsize=(11,5))
for k in indikator:
    plt.plot(harian["date"], harian[k], label=k, linewidth=1)
plt.title("Tren harian indikator kesehatan")
plt.xlabel("Tanggal"); plt.ylabel("Jumlah kasus tercatat")
plt.legend(); plt.tight_layout(); plt.show()


## Tren bulanan

In [ ]:
data["bulan"] = data["date"].dt.to_period("M").astype(str)
bulanan = data.groupby("bulan", as_index=False)[indikator].sum(min_count=1)
display(bulanan.head())
bulanan.to_csv(OUTPUT / "03_tren_bulanan.csv", index=False)
bulanan.set_index("bulan")[indikator].plot(figsize=(11,5))
plt.title("Tren bulanan indikator kesehatan"); plt.xlabel("Bulan"); plt.ylabel("Jumlah kasus tercatat"); plt.xticks(rotation=65); plt.tight_layout(); plt.show()


## Perbandingan wilayah

In [ ]:
wilayah = data.groupby(["provinsi", "kota_kabupaten"], as_index=False).agg(**{k:(k,"sum") for k in indikator}, jumlah_rumah_sakit=("hospital_id","nunique"), jumlah_catatan=("hospital_id","size"), jumlah_hari=("date","nunique"))
wilayah = wilayah.sort_values("resp_admissions_under5", ascending=False)
display(wilayah.head(15))
wilayah.to_csv(OUTPUT / "04_perbandingan_wilayah.csv", index=False)
wilayah.head(12).sort_values("resp_admissions_under5").plot.barh(x="kota_kabupaten", y="resp_admissions_under5", legend=False, figsize=(9,6))
plt.title("12 wilayah dengan total kasus pernapasan balita tertinggi (tercatat)"); plt.xlabel("Jumlah kasus"); plt.tight_layout(); plt.show()


## Kelompok usia

In [ ]:
usia = pd.DataFrame({"kelompok": ["Balita (<5 tahun)", "Lansia (>60 tahun)"], "total_rawat_inap_pernapasan": [data["resp_admissions_under5"].sum(), data["resp_admissions_over60"].sum()]})
display(usia)
usia.to_csv(OUTPUT / "05_perbandingan_usia.csv", index=False)
usia.plot.bar(x="kelompok", y="total_rawat_inap_pernapasan", legend=False, figsize=(7,4))
plt.ylabel("Kasus tercatat"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()


## Data per rumah sakit

In [ ]:
rs_harian = data.groupby(["date","hospital_id","provinsi","kota_kabupaten"], as_index=False)[indikator].sum(min_count=1)
rs_harian.to_csv(OUTPUT / "06_kesehatan_rs_harian.csv", index=False)
display(rs_harian.head())


In [ ]:
audit.to_csv(OUTPUT / "07_missing_indikator.csv", index=True)
catatan = pd.DataFrame({"aspek": ["Periode", "Indikator", "Missing", "Perbandingan wilayah", "Integrasi AQMS", "Interpretasi"], "catatan": ["Analisis utama mulai 2024-01-01; data sumber tidak diubah", "Empat indikator dihitung terpisah, tidak dijumlah menjadi satu total", "Nilai kosong tidak diganti nol", "Total kasus dipengaruhi banyaknya RS dan hari tercatat", "Belum dilakukan; perlu kesesuaian tanggal, wilayah, dan kelengkapan AQMS", "Hasil deskriptif, bukan bukti kausalitas"]})
catatan.to_csv(OUTPUT / "08_catatan_analisis.csv", index=False)
print("Selesai. Output tersimpan di:", OUTPUT.resolve())
print("File:", [p.name for p in OUTPUT.glob("*.csv")])
